# Desafio Semana 1 — Visão Computacional
**Residência em IA | UniSENAI — Prof. Matheus Vanzan**

Construção e avaliação de uma CNN para classificação de imagens de satélite usando o dataset **EuroSAT/RGB**, comparando experimentos com e sem filtros de pré-processamento (OpenCV).


## Seção 0 — Setup e Instalações

In [ ]:
!pip install -q tensorflow-datasets

In [ ]:
import os
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers

print("TF version:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices('GPU'))

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## Seção 1 — Carregamento do Dataset (EuroSAT)

EuroSAT/RGB contém 27.000 imagens 64×64 RGB do Sentinel-2 distribuídas em 10 classes de uso do solo. O TFDS só expõe o split `train`, então fatiamos por porcentagem em treino/val/teste (70/15/15) diretamente pela sintaxe de subsplits — mais limpo e econômico do que carregar tudo em memória para usar `train_test_split`.

> **Nota de compatibilidade:** O servidor original do EuroSAT (madm.dfki.de) frequentemente retorna 403. O parâmetro  faz o TFDS baixar diretamente do bucket público  do Google, que está sempre disponível no Colab sem autenticação adicional.

In [ ]:
DATASET_NAME = 'eurosat/rgb'

# O servidor original (madm.dfki.de) costuma retornar 403.
# try_gcs=True faz o TFDS buscar diretamente no bucket público do Google
# (gs://tfds-data/datasets/) sem passar pelo site de origem.
(ds_train_raw, ds_val_raw, ds_test_raw), info = tfds.load(
    DATASET_NAME,
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    with_info=True,
    as_supervised=True,
    try_gcs=True,       # <-- usa GCS quando o servidor original bloqueia
    download=True,
)

In [ ]:
NUM_CLASSES = info.features['label'].num_classes
CLASS_NAMES = info.features['label'].names
IMG_SIZE    = 64
BATCH_SIZE  = 32

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"IMG_SIZE={IMG_SIZE} | BATCH_SIZE={BATCH_SIZE}")

## Seção 2 — Exploração Inicial

In [ ]:
print(info)
total = info.splits['train'].num_examples
print(f"\nTotal de exemplos: {total}")
print(f"  -> Treino:  {info.splits['train[:70%]'].num_examples}")
print(f"  -> Val:     {info.splits['train[70%:85%]'].num_examples}")
print(f"  -> Teste:   {info.splits['train[85%:]'].num_examples}")

### 2.2 Visualização — 2 amostras por classe

In [ ]:
samples_per_class = {i: [] for i in range(NUM_CLASSES)}
for img, label in tfds.as_numpy(ds_train_raw):
    label = int(label)
    if len(samples_per_class[label]) < 2:
        samples_per_class[label].append(img)
    if all(len(v) == 2 for v in samples_per_class.values()):
        break

plt.figure(figsize=(20, 10))
for class_idx, imgs in samples_per_class.items():
    for j, img in enumerate(imgs):
        pos = class_idx * 2 + j + 1
        plt.subplot(NUM_CLASSES, 2, pos)
        plt.imshow(img)
        plt.title(CLASS_NAMES[class_idx], fontsize=8)
        plt.axis('off')
plt.suptitle('EuroSAT — 2 amostras por classe', fontsize=14)
plt.tight_layout()
plt.show()

### 2.3 Distribuição de Classes

In [ ]:
class_counts = {name: 0 for name in CLASS_NAMES}
for _, label in tfds.as_numpy(ds_train_raw):
    class_counts[CLASS_NAMES[int(label)]] += 1

plt.figure(figsize=(12, 4))
plt.bar(class_counts.keys(), class_counts.values(), color='steelblue')
plt.xticks(rotation=30, ha='right')
plt.title('Distribuição de classes — EuroSAT (train)')
plt.ylabel('Número de imagens')
plt.tight_layout()
plt.show()

print(class_counts)

## Seção 3 — Pré-processamento e Pipeline

### Por que esses filtros fazem sentido para imagens de satélite?

- **Grayscale:** remove a cor — força o modelo a depender de textura/forma. Bom teste de quanto a cor importa em EuroSAT (espera-se que importe bastante).
- **Canny (bordas):** destaca contornos entre regiões de uso do solo (margens de rio, perímetros urbanos, divisões de cultivo). Conexão direta com a Aula 2.
- **Sharpen (nitidez):** realça detalhes finos como estradas, telhados e divisões de talhões — potencialmente útil para Highway, Industrial, AnnualCrop.


In [ ]:
def apply_grayscale(img_np):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    return gray[..., np.newaxis]

def apply_canny(img_np, low=50, high=150):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 1.0)
    edges = cv2.Canny(blurred, low, high)
    return edges[..., np.newaxis]

def apply_sharpen(img_np):
    kernel = np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0]
    ], dtype=np.float32)
    return cv2.filter2D(img_np, -1, kernel)

### 3.3 Pipeline reutilizável `build_dataset_with_filter`

`tf.py_function` apaga o shape estático, então restauramos com `set_shape` para que o `model.fit` saiba a dimensionalidade do tensor. A função aceita qualquer filtro OpenCV via injeção de dependência (`filter_fn`).

In [ ]:
def make_filter_map(filter_fn, out_channels):
    def preprocess_with_filter(img, label):
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.uint8)
        if filter_fn is not None:
            img = tf.py_function(
                func=lambda x: filter_fn(x.numpy()),
                inp=[img],
                Tout=tf.uint8,
            )
        img.set_shape([IMG_SIZE, IMG_SIZE, out_channels])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label
    return preprocess_with_filter


def build_dataset_with_filter(ds_raw, filter_fn, out_channels,
                              batch_size=BATCH_SIZE, shuffle=False):
    fn = make_filter_map(filter_fn, out_channels)
    ds = ds_raw.map(fn, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def build_dataset(ds_raw, batch_size=BATCH_SIZE, shuffle=False):
    """Pipeline sem filtro (baseline): apenas resize + normalização."""
    return build_dataset_with_filter(ds_raw, filter_fn=None, out_channels=3,
                                     batch_size=batch_size, shuffle=shuffle)

### 3.4 Visualização dos filtros aplicados

In [ ]:
sample_img_np = None
for img, _ in tfds.as_numpy(ds_train_raw.take(1)):
    sample_img_np = img

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

axes[0].imshow(sample_img_np)
axes[0].set_title(f'Original\n{sample_img_np.shape}')
axes[0].axis('off')

gray = apply_grayscale(sample_img_np)
axes[1].imshow(gray[..., 0], cmap='gray')
axes[1].set_title(f'Grayscale\n{gray.shape}')
axes[1].axis('off')

canny = apply_canny(sample_img_np)
axes[2].imshow(canny[..., 0], cmap='gray')
axes[2].set_title(f'Canny\n{canny.shape}')
axes[2].axis('off')

sharp = apply_sharpen(sample_img_np)
axes[3].imshow(sharp)
axes[3].set_title(f'Sharpen\n{sharp.shape}')
axes[3].axis('off')

gray_cv = cv2.cvtColor(sample_img_np, cv2.COLOR_RGB2GRAY)
sx = cv2.Sobel(gray_cv, cv2.CV_64F, 1, 0, ksize=3)
sy = cv2.Sobel(gray_cv, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = cv2.convertScaleAbs(cv2.magnitude(sx, sy))
axes[4].imshow(sobel_mag, cmap='gray')
axes[4].set_title('Sobel Mag\n(visual only)')
axes[4].axis('off')

plt.suptitle('Efeito dos filtros — EuroSAT sample', fontsize=13)
plt.tight_layout()
plt.show()

## Seção 4 — Definição da CNN

Arquitetura única (3 blocos Conv+Pool → Flatten → Dense) reaproveitada por todos os experimentos para garantir comparação justa — a única variável que muda entre rodadas é o filtro de entrada (ou o número de épocas).

In [ ]:
def build_cnn(input_shape, num_classes=NUM_CLASSES):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax'),
    ], name='eurosat_cnn')

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

In [ ]:
def run_experiment(name, ds_train, ds_val, ds_test,
                   input_shape, epochs=5):
    print(f"\n{'='*50}")
    print(f"  Experimento: {name}")
    print(f"  Input shape: {input_shape} | Épocas: {epochs}")
    print(f"{'='*50}")

    model = build_cnn(input_shape)
    model.summary()

    history = model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=epochs,
        verbose=1,
    )

    test_loss, test_acc = model.evaluate(ds_test, verbose=0)
    print(f"\n-> Acurácia no teste: {100*test_acc:.2f}%")

    return {
        'name':      name,
        'history':   history,
        'test_loss': test_loss,
        'test_acc':  test_acc,
        'epochs':    epochs,
        'model':     model,
    }

## Seção 5 — Experimentos

| # | Nome | Filtro | Épocas | Input | Pergunta respondida |
|---|---|---|---|---|---|
| 1 | Baseline | — | 5 | (64,64,3) | Referência |
| 2 | Mais épocas | — | 10 | (64,64,3) | Efeito de épocas |
| 3 | Grayscale | Grayscale | 5 | (64,64,1) | Filtro vs baseline |
| 4 | Canny | Canny | 5 | (64,64,1) | Filtro de borda |
| 5 | Sharpen | Sharpen | 5 | (64,64,3) | Filtro de nitidez |


### Experimento 1 — Baseline (5 épocas)

In [ ]:
ds_train_base = build_dataset(ds_train_raw, shuffle=True)
ds_val_base   = build_dataset(ds_val_raw)
ds_test_base  = build_dataset(ds_test_raw)

exp1 = run_experiment(
    name='Exp1 — Baseline (5 épocas)',
    ds_train=ds_train_base, ds_val=ds_val_base, ds_test=ds_test_base,
    input_shape=(IMG_SIZE, IMG_SIZE, 3), epochs=5,
)

### Experimento 2 — Baseline com 10 épocas

In [ ]:
exp2 = run_experiment(
    name='Exp2 — Baseline (10 épocas)',
    ds_train=ds_train_base, ds_val=ds_val_base, ds_test=ds_test_base,
    input_shape=(IMG_SIZE, IMG_SIZE, 3), epochs=10,
)

### Experimento 3 — Grayscale

In [ ]:
ds_train_gray = build_dataset_with_filter(ds_train_raw, apply_grayscale, out_channels=1, shuffle=True)
ds_val_gray   = build_dataset_with_filter(ds_val_raw,   apply_grayscale, out_channels=1)
ds_test_gray  = build_dataset_with_filter(ds_test_raw,  apply_grayscale, out_channels=1)

exp3 = run_experiment(
    name='Exp3 — Grayscale (5 épocas)',
    ds_train=ds_train_gray, ds_val=ds_val_gray, ds_test=ds_test_gray,
    input_shape=(IMG_SIZE, IMG_SIZE, 1), epochs=5,
)

### Experimento 4 — Canny

In [ ]:
ds_train_canny = build_dataset_with_filter(ds_train_raw, apply_canny, out_channels=1, shuffle=True)
ds_val_canny   = build_dataset_with_filter(ds_val_raw,   apply_canny, out_channels=1)
ds_test_canny  = build_dataset_with_filter(ds_test_raw,  apply_canny, out_channels=1)

exp4 = run_experiment(
    name='Exp4 — Canny (5 épocas)',
    ds_train=ds_train_canny, ds_val=ds_val_canny, ds_test=ds_test_canny,
    input_shape=(IMG_SIZE, IMG_SIZE, 1), epochs=5,
)

### Experimento 5 — Sharpen

In [ ]:
ds_train_sharp = build_dataset_with_filter(ds_train_raw, apply_sharpen, out_channels=3, shuffle=True)
ds_val_sharp   = build_dataset_with_filter(ds_val_raw,   apply_sharpen, out_channels=3)
ds_test_sharp  = build_dataset_with_filter(ds_test_raw,  apply_sharpen, out_channels=3)

exp5 = run_experiment(
    name='Exp5 — Sharpen (5 épocas)',
    ds_train=ds_train_sharp, ds_val=ds_val_sharp, ds_test=ds_test_sharp,
    input_shape=(IMG_SIZE, IMG_SIZE, 3), epochs=5,
)

## Seção 6 — Tabela de Resultados (7.1)

In [ ]:
experiments = [exp1, exp2, exp3, exp4, exp5]

print(f"{'Experimento':<35} {'Épocas':>6} {'Input Shape':>14} {'Acurácia Teste':>16}")
print("-" * 75)
for exp in experiments:
    shape_str = str(exp['model'].input_shape[1:])
    print(f"{exp['name']:<35} {exp['epochs']:>6} {shape_str:>14} {100*exp['test_acc']:>15.2f}%")

### Curvas de Treinamento

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, exp in enumerate(experiments):
    ax = axes[i]
    h = exp['history'].history
    ax.plot(h['accuracy'],     label='Train Acc')
    ax.plot(h['val_accuracy'], label='Val Acc')
    ax.set_title(exp['name'])
    ax.set_xlabel('Época')
    ax.set_ylabel('Acurácia')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])

axes[5].set_visible(False)
plt.suptitle('Curvas de Treinamento — Todos os Experimentos', fontsize=14)
plt.tight_layout()
plt.show()

### Gráfico Comparativo de Acurácia

In [ ]:
names = [exp['name'].replace(' — ', '\n') for exp in experiments]
accs  = [100 * exp['test_acc'] for exp in experiments]
colors = ['steelblue', 'steelblue', 'darkorange', 'green', 'purple']

plt.figure(figsize=(12, 5))
bars = plt.bar(names, accs, color=colors, alpha=0.8, edgecolor='black')
plt.ylabel('Acurácia no Teste (%)')
plt.title('Comparativo de Acurácia — EuroSAT')
plt.ylim([0, 100])
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## Seção 7 — Análise Comparativa (7.2)

> Os números abaixo devem ser preenchidos a partir da execução real do notebook. As respostas comentam o que esperar e como interpretar.

### 7.2.1 — Filtros vs Sem Filtros

Comparar **Exp1 (RGB sem filtro)** com **Exp3 (Grayscale)**, **Exp4 (Canny)** e **Exp5 (Sharpen)**.

- O baseline RGB tende a ser o melhor: em EuroSAT, **cor é altamente discriminante** (vegetação verde, água azul, área urbana cinza, solo exposto marrom). Qualquer filtro que descarte cor (Grayscale, Canny) perde informação útil.
- **Canny** tende a ser o pior — descarta cor *e* textura, deixando apenas contornos. Para satélite isso é especialmente prejudicial, porque classes como `Forest`, `Pasture` e `HerbaceousVegetation` se distinguem mais por textura/cor do que por bordas nítidas.
- **Sharpen** mantém os 3 canais e só realça alta-frequência — costuma ficar próximo do baseline (ligeira melhora ou neutro), podendo ajudar em classes urbanas (`Highway`, `Industrial`).

### 7.2.2 — Efeito do Número de Épocas

Comparar **Exp1 (5 épocas)** vs **Exp2 (10 épocas)**.

- Com 5 épocas o modelo costuma ainda estar subajustado. 10 épocas geralmente trazem ganho, mas com retornos decrescentes.
- Observar as curvas de `val_accuracy`: se ela já estava saturando no Exp1, o ganho em Exp2 será pequeno; se ainda subia, o ganho será maior. Atenção a sinais de overfitting (train_acc subindo e val_acc estagnando).

### 7.2.3 — Tamanho da Imagem

EuroSAT já vem em 64×64 (resolução nativa do RGB derivado do Sentinel-2).

- Reduzir para 32×32 perderia detalhes finos (estradas, divisões de talhões) e provavelmente prejudicaria `Highway` e `Industrial`.
- Aumentar para 128×128 com `tf.image.resize` apenas interpola — não cria informação. Aumenta custo computacional sem ganho real. Faria sentido apenas se fôssemos usar uma backbone pré-treinada que exige entradas maiores.

### 7.2.4 — O que tentaria para melhorar

- **Data augmentation** apropriado para satélite: `RandomFlip` horizontal/vertical e `RandomRotation` (imagens aéreas são invariantes a rotação — diferente de fotos comuns).
- **BatchNormalization** entre Conv e ativação para acelerar e estabilizar o treino.
- **Dropout** (~0.3–0.5) antes da Dense final para reduzir overfitting.
- **Learning rate scheduling** (`ReduceLROnPlateau` ou cosine decay).
- **Transfer learning** com backbone pré-treinada (EfficientNetB0, MobileNetV2) — provavelmente o maior salto de acurácia possível.

### 7.2.5 — Conclusão sobre Pré-processamento

Para EuroSAT, **menos pré-processamento é mais**: a CNN aprende sozinha as features relevantes a partir da entrada RGB crua. Filtros clássicos como Canny, que descartam canais de informação, *atrapalham* o modelo quando a tarefa depende justamente daquela informação descartada.

Isso reforça o ponto da Aula 2: filtros tradicionais são úteis quando funcionam como **engenharia de feature direcionada** (ex.: contagem de bordas para inspeção industrial), mas em problemas de classificação onde a rede pode aprender features por conta própria, o melhor pré-processamento tende a ser apenas a normalização.
